# Train Ensemble² — 12-Channel Early Fusion Model

This notebook trains the single 12-channel model that concatenates
RGB + Lab + HSV + YCrCb as its input (Ensemble² in the paper).

**Prerequisites:** `retinal_colab_data_bundle.zip` must exist in your Drive.
The zip is extracted to Colab's local disk for fast I/O during training.
The final model checkpoint and predictions are written back to Drive.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. User Settings

Replace every `CHANGE_ME` value before running.

In [ ]:
from pathlib import Path

# Full path to the zip file on Drive
DRIVE_ZIP_PATH = Path('/content/drive/MyDrive/CHANGE_ME/retinal_colab_data_bundle.zip')

# Repository (same as main notebook)
REPO_URL = 'https://github.com/CHANGE_ME/retinal-color-transfer.git'
REPO_REF = 'main'

# Drive folder where the trained model will be saved
DRIVE_MODEL_ROOT = Path('/content/drive/MyDrive/CHANGE_ME/retinal-color-transfer-run/models')

# Training hyperparameters (keep consistent with main study)
EPOCHS        = 80
BATCH_SIZE    = 32
NUM_WORKERS   = 2
MIXED_PRECISION = 'fp16'
SEED          = 42

# Local Colab paths (fast SSD, no need to change)
LOCAL_DATA_DIR  = Path('/content/local_data')
REPO_DIR        = Path('/content/retinal-color-transfer')
LOCAL_MODEL_DIR = Path('/content/fusion_model')
MODEL_NAME      = 'rgb_lab_hsv_ycrcb_seed42'

print('Settings OK')

## 3. Validate Settings

In [ ]:
for name, val in [('DRIVE_ZIP_PATH', str(DRIVE_ZIP_PATH)),
                   ('REPO_URL', REPO_URL),
                   ('DRIVE_MODEL_ROOT', str(DRIVE_MODEL_ROOT))]:
    if 'CHANGE_ME' in val:
        raise ValueError(f'Replace the placeholder in: {name}')

if not DRIVE_ZIP_PATH.is_file():
    raise FileNotFoundError(f'Zip not found: {DRIVE_ZIP_PATH}')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA required. Go to Runtime → Change runtime type → GPU.')

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Zip: {DRIVE_ZIP_PATH} ({DRIVE_ZIP_PATH.stat().st_size / 1e9:.1f} GB)')

## 4. Copy Zip to Local Disk and Extract

Colab's local SSD is ~5× faster than Drive for random reads.
We copy the zip once and extract only the four cache families needed.

In [ ]:
import shutil, subprocess, time

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
local_zip = LOCAL_DATA_DIR / 'bundle.zip'

# Only copy if not already present
if not local_zip.is_file():
    print('Copying zip from Drive to local disk...')
    t0 = time.time()
    shutil.copy2(DRIVE_ZIP_PATH, local_zip)
    print(f'Copied in {time.time()-t0:.0f}s')
else:
    print('Zip already on local disk, skipping copy.')

# Extract only the 4 cache dirs we need
NEEDED_PREFIXES = ['caches/rgb/rgb/', 'caches/lab/lab/',
                   'caches/hsv/hsv/', 'caches/ycrcb/ycrcb/']

# Check if already extracted
all_extracted = all(
    (LOCAL_DATA_DIR / p).is_dir() for p in NEEDED_PREFIXES
)

if not all_extracted:
    print('Extracting cache directories...')
    t0 = time.time()
    # Extract only matching prefixes
    patterns = ' '.join(f"'{p}*'" for p in NEEDED_PREFIXES)
    subprocess.run(
        f'unzip -q -o {local_zip} {patterns} -d {LOCAL_DATA_DIR}',
        shell=True, check=True
    )
    print(f'Extracted in {time.time()-t0:.0f}s')
else:
    print('Cache directories already extracted.')

CACHE_ROOT = LOCAL_DATA_DIR / 'caches'
print(f'Cache root: {CACHE_ROOT}')

# Quick sanity check
for rep in ['rgb/rgb', 'lab/lab', 'hsv/hsv', 'ycrcb/ycrcb']:
    count = len(list((CACHE_ROOT / rep).glob('*.png')))
    print(f'  {rep}: {count} images')

## 5. Clone and Install Repository

In [ ]:
import importlib, os, shutil, subprocess, sys

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ['git', 'clone', '--quiet', '--depth', '1',
     '--branch', REPO_REF, REPO_URL, str(REPO_DIR)],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)],
    check=True,
)

src_dir = str(REPO_DIR / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
importlib.invalidate_caches()
os.chdir(REPO_DIR)

import retinal_color_transfer
print('Repo:', Path.cwd())
print('Package:', retinal_color_transfer.__file__)

## 6. Train the 12-Channel Fusion Model

The script:
- Reads RGB, Lab, HSV, YCrCb caches and concatenates them into 12-ch tensors
- Initialises ResNet-50 with conv1 adapted to 12 channels (pretrained weights ×4, scaled by 1/4)
- Trains with the same protocol as all other study models (80 ep, AdamW, LDS, etc.)
- Resumes automatically if `latest_checkpoint.pt` is present
- Saves `best_checkpoint.pt`, `training_history.csv`, and `predictions.csv`

In [ ]:
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        sys.executable, 'scripts/train_fusion12ch.py',
        '--cache-root',        str(CACHE_ROOT),
        '--model-dir',         str(LOCAL_MODEL_DIR),
        '--device',            'cuda',
        '--epochs',            str(EPOCHS),
        '--batch-size',        str(BATCH_SIZE),
        '--num-workers',       str(NUM_WORKERS),
        '--seed',              str(SEED),
        '--mixed-precision',   MIXED_PRECISION,
        '--allow-weight-download',
    ],
    check=True,
)

## 7. Save Model Artifacts to Drive

In [ ]:
drive_dest = DRIVE_MODEL_ROOT / 'fusion' / MODEL_NAME
drive_dest.mkdir(parents=True, exist_ok=True)

EXPECTED_FILES = {'best_checkpoint.pt', 'training_history.csv', 'predictions.csv'}
missing = EXPECTED_FILES - {f.name for f in LOCAL_MODEL_DIR.iterdir()}
if missing:
    raise RuntimeError(f'Training did not produce expected files: {missing}')

for fname in EXPECTED_FILES:
    src = LOCAL_MODEL_DIR / fname
    dst = drive_dest / fname
    shutil.copy2(src, dst)
    print(f'Saved: {dst}')

# Quick result summary
import pandas as pd
preds = pd.read_csv(drive_dest / 'predictions.csv')
for split in ['validation', 'test']:
    mae = preds[preds['split'] == split]['absolute_error'].mean()
    print(f'  {split} MAE: {mae:.4f}')

print('\nDone! Model saved to Drive.')